In [ ]:
import os
import json
from google.colab import userdata

# 1. Setup Kaggle Credentials safely
# Make sure you've added KAGGLE_USERNAME and KAGGLE_KEY to the "Keys" (Secrets) tab on the left!
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Create the hidden folder Kaggle expects
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
!chmod 600 ~/.kaggle/kaggle.json

# 2. Install necessary libraries
!pip install -q ultralytics

# 3. Download and Unzip (Your specific commands)
print("Downloading dataset...")
!kaggle datasets download -d sathyakirants/pose-estimation-videos -p ./data
!unzip -q ./data/pose-estimation-videos.zip -d ./data

print("\n✅ Setup Complete! Check the 'data' folder in the sidebar to see your video folders.")


Dataset URL: https://www.kaggle.com/datasets/sathyakirants/pose-estimation-videos
License(s): unknown
100% 6.49G/6.49G [07:33<00:00, 15.4MB/s]


✅ Setup Complete! Check the 'data' folder in the sidebar to see your video folders.


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from pathlib import Path
from tqdm.auto import tqdm # Progress bar

# 1. Load Pose model on GPU
model = YOLO('yolov8n-pose.pt')

# 2. Setup Paths based on your screenshot
DATA_DIR = Path('./data/UCF-101')
SEQ_LEN = 30 # We will normalize every video to 30 frames

X_data = []
y_labels = []

# 3. Get the list of action folders
actions = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Found {len(actions)} Action Classes.")

# Loop through each action folder
for idx, action in enumerate(actions):
    action_path = DATA_DIR / action

    # UCF-101 often uses .avi files
    video_paths = list(action_path.glob('*.avi')) + list(action_path.glob('*.mp4'))

    if not video_paths:
        continue

    print(f"\n🎬 Processing {action} ({len(video_paths)} videos)")

    # Use tqdm to see progress for each folder
    for v_path in tqdm(video_paths):
        cap = cv2.VideoCapture(str(v_path))
        temp_seq = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            results = model(frame, verbose=False, device=device_to_use)

            # 1. Check if the model actually found ANY keypoints
            if results[0].keypoints is not None and len(results[0].keypoints.xy) > 0:

                # 2. Get the first detected person's keypoints
                kp = results[0].keypoints.xy[0].cpu().numpy()

                # 3. Double check the keypoints aren't just empty/zeros
                if kp.shape[0] > 0:
                    joints = kp.flatten()
                    temp_seq.append(joints)

        cap.release()

        # If video has enough frames, take a 30-frame slice from the middle
        if len(temp_seq) >= SEQ_LEN:
            mid = len(temp_seq) // 2
            start = mid - (SEQ_LEN // 2)
            X_data.append(temp_seq[start : start + SEQ_LEN])
            y_labels.append(idx)

# 4. Final Save
if len(X_data) > 0:
    np.save('X_train.npy', np.array(X_data))
    np.save('y_train.npy', np.array(y_labels))
    np.save('action_names.npy', np.array(actions)) # Save names for later
    print(f"\n✅ SUCCESS! Processed {len(X_data)} sequences total.")
else:
    print("\n❌ ERROR: No videos were processed. Check if the folder names match!")

Found 101 Action Classes.

🎬 Processing ApplyEyeMakeup (145 videos)


  0%|          | 0/145 [00:00<?, ?it/s]


🎬 Processing ApplyLipstick (114 videos)


  0%|          | 0/114 [00:00<?, ?it/s]

IndexError: index 0 is out of bounds for dimension 0 with size 0